### Importing necessary modules

In [1]:
import pandas as pd                                                                     # type: ignore
import matplotlib.pyplot as plt                                                         # type: ignore
import seaborn as sns                                                                   # type: ignore
import numpy as np                                                                      # type: ignore
from  sklearn import metrics                                                            # type: ignore
from sklearn.preprocessing import StandardScaler                                        # type: ignore
from sklearn.metrics import roc_auc_score, roc_curve, brier_score_loss                  # type: ignore
from sklearn.metrics import average_precision_score                                     # type: ignore
import tensorflow as tf                                                                 # type: ignore
from sklearn.model_selection import train_test_split                                    # type: ignore
import matplotlib as mpl                                                                # type: ignore
colors = plt.rcParams['axes.prop_cycle'].by_key()['color']
mpl.rcParams['figure.figsize'] = (9, 7)
from joblib import dump, load                                                           # type: ignore
import random
import cartopy                                                                          # type: ignore
from matplotlib.offsetbox import AnchoredText                                           # type: ignore
import cartopy.crs as ccrs                                                              # type: ignore
import cartopy.feature as cfeature                                                      # type: ignore

2024-06-13 10:30:30.809876: I external/local_tsl/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2024-06-13 10:30:30.814567: I external/local_tsl/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2024-06-13 10:30:30.872319: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2024-06-13 10:30:32.207472: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


In [2]:
Dakar_lon = -17.467686
Dakar_lat = 14.716677

In [3]:
import sys
sys.path.insert(1, "/home/mendrika/mendrika-phd/codes/nflics")
import nflics  

### Importing dataset

In [4]:
test_data = pd.read_csv("/home/mendrika/mendrika-phd/codes/nflics/output/SM2024-06-13/raw-artificial-data-two-storms-same-size-and-wp-different-t0-and-location.csv", index_col=False)
raw_test_data = test_data.copy()

In [5]:
pcX0 = pd.read_csv(f"/home/mendrika/mendrika-phd/codes/nflics/output/SM2024-06-13/pcX0-two-storms-same-size-and-wp-different-t0-and-location.csv", index_col=None)
to_keep = pcX0["index"].values
to_drop = [i for i in range(len(test_data)) if i not in to_keep]

In [6]:
test_data = test_data.drop(index=to_drop)
raw_test_data = raw_test_data.drop(index=to_drop)

#### Choosing field to include in the data

In [7]:
# Presence or absence of convection in Dakar at time t+1
target_index = "Cb_Dakar"

# Input at time t0
t0 = "year,month,day,hour,minute,"

# latitude and longitude, wavelet power, storm size and distance to Dakar
location = ""
wavelet_power = ""
storm_size = ""
distance = ""
for i in range(1,6):
    location += f"lat{i},lon{i},"
    wavelet_power += f"wp{i},"
    distance += f"ds{i},"
    storm_size += f"size{i},"

# combining all fields to form the feature input
features = t0 + location + wavelet_power + storm_size + distance
field =  features + target_index
field = field.split(',')

### Exploratory data analysis

In [8]:
def log_transform(df, keys):
    df_copy = df.copy()    
    for key in keys:
        transformed = []
        for i in df_copy[key]:
            if i > 0:
                transformed.append(np.log(i))
            else:
                transformed.append(np.log(i+1e-8))    
        df_copy[key] = transformed
    return df_copy

In [9]:
def sqrt_transform(df, keys):
    df_copy = df.copy()    
    for key in keys:
        transformed = []
        for i in df_copy[key]:
            if i >= 0:
                transformed.append(np.sqrt(i)) 
        df_copy[key] = transformed
    return df_copy

In [10]:
to_scale = wavelet_power + storm_size + distance

In [11]:
test_data = log_transform(test_data, to_scale.split(',')[:-1])
test_data = sqrt_transform(test_data, t0.split(',')[:-1])

In [12]:
model = tf.keras.models.load_model("/home/mendrika/mendrika-phd/codes/nflics/model-ML-Dakar/best_model_new_format_Dakar_nearest5_corrected.keras")

In [13]:
scaler = load('std_scaler_best_model_new_format_Dakar_nearest5_corrected.bin')

In [14]:
x_test = test_data[features.split(",")[:-1]]
x_test_std = scaler.transform(x_test)

In [30]:
y_pred_nn = model.predict(x_test_std)

31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step


In [31]:
y_pred_nn = y_pred_nn.flatten()

### Unconditional

In [17]:
pc = pd.read_csv("/home/mendrika/mendrika-phd/codes/nflics/unconditional-probability/dataset-pc-31.csv")

In [35]:
Cb_Dakar_unc = []
for i in range(len(raw_test_data)):
    minute = raw_test_data["minute"].values[i]    
    hour = raw_test_data["hour"].values[i] + 1
    if hour != 24:    
        index_hour = pc["hour"].values == hour
    else:
        index_hour = pc["hour"].values == 0
    index_minute = pc["minute"].values == minute
    index = index_hour * index_minute
    Cb_Dakar_unc.append(pc["probability"].values[index])
y_pred_clim = np.array(Cb_Dakar_unc).flatten()

In [26]:
y_pred_nflics = pcX0["pcX0"].values

In [36]:
index_comp_nflics = y_pred_nflics < y_pred_nn
index_comp_clim = y_pred_clim < y_pred_nn

In [37]:
df_selected_nflics = raw_test_data.loc[index_comp_nflics]
df_selected_clim = raw_test_data.loc[index_comp_clim]

In [38]:
df_selected_nflics

,year,month,day,hour,minute,lat1,lon1,lat2,lon2,lat3,...,size1,size2,size3,size4,size5,ds1,ds2,ds3,ds4,ds5
20,2008,8,3,13,0,14.069591,-17.292319,14.224282,-10.062736,14,...,5000.0,5000.0,0,0,0,22.2036,251.3901,430,430,430
70,2010,8,20,21,0,17.937526,-15.713984,16.087302,-17.627783,14,...,5000.0,5000.0,0,0,0,130.1730,47.0106,430,430,430
75,2008,7,24,11,0,10.079470,-11.681875,12.202506,-12.258080,14,...,5000.0,5000.0,0,0,0,246.7590,190.5151,430,430,430
99,2006,8,14,21,0,14.792024,-17.996106,14.014509,-13.986191,14,...,5000.0,5000.0,0,0,0,18.2483,117.2774,430,430,430
108,2017,8,27,10,0,12.638838,-10.385815,10.243078,-11.774612,14,...,5000.0,5000.0,0,0,0,247.1214,241.2302,430,430,430
119,2014,8,2,10,0,17.962123,-17.839497,10.865586,-12.442059,14,...,5000.0,5000.0,0,0,0,111.0045,209.4684,430,430,430
152,2014,8,21,16,0,13.758409,-17.138268,14.533483,-10.161784,14,...,5000.0,5000.0,0,0,0,33.9559,248.0181,430,430,430
188,2014,7,11,7,0,14.946904,-17.026998,15.098594,-17.533666,14,...,5000.0,5000.0,0,0,0,17.0000,13.0384,430,430,430
200,2017,7,12,5,0,17.670745,-10.711488,10.957398,-17.084251,14,...,5000.0,5000.0,0,0,0,258.3041,132.0152,430,430,430
207,2016,6,16,5,0,11.913448,-16.447004,15.777255,-10.029201,14,...,5000.0,5000.0,0,0,0,101.3903,258.1182,430,430,430


In [40]:
df_selected_clim

,year,month,day,hour,minute,lat1,lon1,lat2,lon2,lat3,...,size1,size2,size3,size4,size5,ds1,ds2,ds3,ds4,ds5
20,2008,8,3,13,0,14.069591,-17.292319,14.224282,-10.062736,14,...,5000.0,5000.0,0,0,0,22.2036,251.3901,430,430,430
43,2006,7,6,11,0,13.648312,-16.722110,17.693135,-11.299074,14,...,5000.0,5000.0,0,0,0,41.6773,240.1687,430,430,430
99,2006,8,14,21,0,14.792024,-17.996106,14.014509,-13.986191,14,...,5000.0,5000.0,0,0,0,18.2483,117.2774,430,430,430
106,2005,8,10,20,0,14.450480,-16.503082,15.448891,-17.889227,14,...,5000.0,5000.0,0,0,0,32.2800,27.7308,430,430,430
125,2009,6,2,21,0,14.420235,-16.710983,14.548392,-14.114321,14,...,5000.0,5000.0,0,0,0,26.0000,112.0714,430,430,430
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
891,2009,8,24,12,0,15.329425,-15.782273,15.288948,-12.412143,14,...,5000.0,5000.0,0,0,0,62.0322,173.4013,430,430,430
907,2008,7,5,8,0,14.001416,-17.051174,12.358573,-12.680995,14,...,5000.0,5000.0,0,0,0,26.4008,175.7754,430,430,430
938,2007,7,24,9,0,15.500829,-17.915916,15.927838,-13.705045,14,...,5000.0,5000.0,0,0,0,29.9666,136.2975,430,430,430
978,2011,8,28,12,0,15.848584,-16.296669,13.931740,-16.449457,14,...,5000.0,5000.0,0,0,0,58.0000,41.1096,430,430,430
